In [2]:
import os
import time
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import errors
from pypdf import PdfReader

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

GENERATION_MODELS = ["gemini-flash-lite-latest", "gemini-flash-latest", "gemini-3.8-flash"]
EMBEDDING_MODEL = "gemini-embedding-001"


def with_retry(operation, max_attempts=5):
    for attempt in range(max_attempts):
        try:
            return operation()
        except errors.APIError as e:
            if e.code == 429:
                print("Quota exceeded, waiting 60 seconds...")
                time.sleep(60)
            elif e.code >= 500:
                wait = 2 ** attempt
                print(f"Server busy, waiting {wait} seconds...")
                time.sleep(wait)
            else:
                raise
    raise Exception("Operation failed, try again in a few minutes.")


def embed(texts):
    result = with_retry(lambda: client.models.embed_content(
        model=EMBEDDING_MODEL, contents=texts))
    return [e.values for e in result.embeddings]


def generate(prompt, rounds=3):
    for r in range(rounds):
        for model in GENERATION_MODELS:
            try:
                return client.models.generate_content(model=model, contents=prompt)
            except errors.APIError as e:
                print(f"{model} unavailable ({e.code})")
        wait = 5 * (r + 1)
        print(f"All models busy, retrying in {wait} seconds...")
        time.sleep(wait)
    raise Exception("No model responded, try again in a few minutes.")


def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


print("Ready!")

Ready!


In [3]:
import re

PDF_PATH = "documents/hr_regulation.pdf"
SECTION_START = re.compile(r"^(\d+(?:\.\d+)*)\.\s+(.*)")
MIN_SECTION_LENGTH = 30


def read_sections(pdf_path):
    reader = PdfReader(pdf_path)
    sections = []
    current = None
    for page_no, page in enumerate(reader.pages, start=1):
        for line in (page.extract_text() or "").split("\n"):
            line = line.strip()
            if not line or re.fullmatch(r"Sayfa \d+", line):
                continue
            match = SECTION_START.match(line)
            if match:
                current = {"number": match.group(1), "text": line, "page": page_no}
                sections.append(current)
            elif current:
                current["text"] += " " + line
    return sections


def short_title(text, max_length=80):
    title = re.split(r"[:;]", text, maxsplit=1)[0]
    return title[:max_length].strip()


def build_chunks(sections):
    titles = {s["number"]: short_title(s["text"]) for s in sections}
    chunks = []
    for index, s in enumerate(sections):
        if len(s["text"]) < MIN_SECTION_LENGTH:
            continue  # skip heading-only sections, their content lives in children
        parts = s["number"].split(".")
        parent_numbers = [".".join(parts[:i]) for i in range(1, len(parts))]
        breadcrumb = " > ".join(titles[n] for n in parent_numbers if n in titles)
        chunks.append({
            "id": f"{index:04d}-{s['number']}",
            "text": f"{breadcrumb}\n{s['text']}" if breadcrumb else s["text"],
            "section": s["number"],
            "page": s["page"],
        })
    return chunks


sections = read_sections(PDF_PATH)
chunks = build_chunks(sections)
print(f"{len(sections)} sections found, {len(chunks)} chunks created.")
print("\nExample chunk:")
print(next(c["text"] for c in chunks if c["section"] == "5.7.5.1.1"))

266 sections found, 238 chunks created.

Example chunk:
5. İSTİHDAM ŞEKİLLERİ, POZİSYONLARIN TESPİTİ VE GRUPLANDIRILMASI > 5.7. Çalışma Saatleri, Fazla Çalışma ve İzinler > 5.7.5. Mazeret izni > 5.7.5.1. Doğum İzni
5.7.5.1.1. Evlenmesi veya eşinin doğum yapması halinde üç gün,


In [4]:
import chromadb

DB_PATH = "chroma_db"
COLLECTION_NAME = "hr_regulation"

db = chromadb.PersistentClient(path=DB_PATH)
collection = db.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

if collection.count() > 0:
    print(f"Collection already has {collection.count()} chunks, skipping indexing.")
else:
    BATCH_SIZE = 20
    for i in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[i:i + BATCH_SIZE]
        collection.add(
            ids=[c["id"] for c in batch],
            documents=[c["text"] for c in batch],
            embeddings=embed([c["text"] for c in batch]),
            metadatas=[{"section": c["section"], "page": c["page"]} for c in batch],
        )
        print(f"{min(i + BATCH_SIZE, len(chunks))}/{len(chunks)} chunks indexed")
        if i + BATCH_SIZE < len(chunks):
            time.sleep(15)  # stay under the free tier limit
    print("Indexing complete.")

20/238 chunks indexed
40/238 chunks indexed
60/238 chunks indexed
80/238 chunks indexed
100/238 chunks indexed
120/238 chunks indexed
140/238 chunks indexed
160/238 chunks indexed
180/238 chunks indexed
200/238 chunks indexed
220/238 chunks indexed
238/238 chunks indexed
Indexing complete.


In [5]:
def search(question, top_k=5, threshold=0.6):
    result = collection.query(query_embeddings=embed(question), n_results=top_k)
    hits = []
    for text, meta, distance in zip(result["documents"][0], result["metadatas"][0], result["distances"][0]):
        similarity = 1 - distance  # Chroma returns cosine distance
        if similarity >= threshold:
            hits.append({"text": text, "section": meta["section"], "page": meta["page"], "score": similarity})
    return hits


def ask(question, top_k=5, threshold=0.6):
    print("Question:", question)
    hits = search(question, top_k, threshold)

    if not hits:
        print("Answer: Bu konuda bilgim yok (no relevant section found).")
        print("-" * 50)
        return

    context = "\n\n".join(f"[Section {h['section']}, Page {h['page']}]\n{h['text']}" for h in hits)
    prompt = f"""You are an HR assistant. Answer ONLY using the context below.
If the answer is not in the context, reply exactly: "Bu konuda bilgim yok."
Answer in Turkish.
If the question has several parts, answer each part separately.
Include every related right or rule in the context, not only the most obvious one.
If a part of the question is not covered by the context, say so for that part.
At the end, list the section numbers you used.

Context:
{context}

Question: {question}"""

    answer = generate(prompt)
    print("Sections found:", [h["section"] for h in hits])
    print("\nAnswer:", answer.text)
    print("-" * 50)


ask("5 yıldır çalışıyorum, kaç gün yıllık izin hakkım var?")
ask("Evlenirsem kaç gün izin alırım?")
ask("Deneme süresi ne kadar?")
ask("Karım doğum yaptı, ikimiz ne kadar izin alabiliriz?")
ask("Yemek kartına ne kadar yükleniyor?")

Question: 5 yıldır çalışıyorum, kaç gün yıllık izin hakkım var?
Sections found: ['5.7.4.1', '5.7.4', '5.7.4.2', '5.7.4.3', '5.7.4.5']

Answer: - Beş yıla kadar (beş yıl dahil) olanlar için on dört iş günü yıllık izin hakkınız vardır.

Kullanılan Bölüm Numaraları: 5.7.4.1, 5.7.4
--------------------------------------------------
Question: Evlenirsem kaç gün izin alırım?
Sections found: ['5.7.5.1.1', '5.7.5.1.3', '5.7.8', '5.7.4.1', '5.7.5.1']

Answer: - Evlenmeniz halinde üç gün izin verilir.
- Çocuğunuzun evlenmesi halinde iki gün izin verilir.

Kullanılan bölüm numaraları: 5.7.5.1.1, 5.7.5.1.3
--------------------------------------------------
Question: Deneme süresi ne kadar?
Sections found: ['5.3.2', '5.3.1', '3.6', '5.3.3', '3.4']

Answer: Deneme süresi 2 aydır.

Kullanılan bölüm numaraları: 5.3.2
--------------------------------------------------
Question: Karım doğum yaptı, ikimiz ne kadar izin alabiliriz?
Sections found: ['5.7.5.1.1', '5.7.8.3', '5.7.5.1', '5.7.5.1.3', '5.7.5.1.

In [7]:
ask("çok heycanlıyım karım gebe, o da burada çalışıyor, biz ne kadar izin alacağız ve karım gebelikten sonra  ayrılacak tazminatını hesaplar mısın")


Question: çok heycanlıyım karım gebe, o da burada çalışıyor, biz ne kadar izin alacağız ve karım gebelikten sonra  ayrılacak tazminatını hesaplar mısın
Sections found: ['5.7.5.1.1', '5.7.5.1', '5.7.8.3', '5.7.5.1.3', '5.7.5.1.2']

Answer: - Eşinizin doğum yapması halinde size üç gün mazeret izni verilir.
- Eşiniz doğum yapmasından önce sekiz hafta ve doğum yaptığı tarihten itibaren sekiz hafta doğum izni alabilir.
- Eşiniz doğum izni süresinden sonra, çocuğu bir yaşına gelinceye kadar günde bir buçuk saat süt izni alabilir. Süt izinleri eşinizin talebi halinde haftalık, aylık veya tamamının toplu olarak kullanılması şeklinde olabilir. Toplu kullanımların hesabında hafta tatili, resmi ve bayram tatillerine rastlayan günlere ait günlük 1,5 saatlik süt izinleri hesaba alınmaz.
- Eşiniz doğum yapan personel olması durumunda, isteği halinde en çok altı aya kadar ücretsiz izin alabilir.
- Eşinizin gebelikten sonra işten ayrıılması durumunda tazminatının hesaplanması bu bağlamda metinde yer a